# Section 0. Notebook purpose and safety

This notebook traces how the block-rendered prompt and Genetic Algorithm (GA) prompt search actually work in the `JOILang-Server` repository.

**Important Safety & Scope Notes:**
- It **does not** run full paper experiments or require five local models.
- **Retrieval pre-mapping** is considered fixed runtime context construction, not a GA mutation target.
- **Core blocks** are always included in the prompt, while **optional blocks** are searchable and evolvable by the GA.
- This is designed to be safely executed interactively for debugging, inspection, and experiment-tracing.

## Configuration Toggles

In [1]:
USE_REAL_MODEL = False
USE_MOCK_LLM = True
USE_ADVISOR = False
SIMULATE_OOM_FEEDBACK = True
SHOW_FULL_PROMPTS = False

MODEL_KEY = "qwen25_coder_7b"
ROW_NO = 1
CATEGORIES = [1, 2]
POPULATION = 4
GENERATIONS = 2
CANDIDATE_K = 1
REPAIR_ATTEMPTS = 0
PROGRESS = "minimal"

# Section 1. Environment and path setup

In [2]:
import sys
import os
import json
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
import hashlib

# Resolve workspace paths
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
VERSION_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = VERSION_ROOT.parent.parent

if str(VERSION_ROOT) not in sys.path:
    sys.path.insert(0, str(VERSION_ROOT))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Python executable: {sys.executable}")
try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError:
    print("CUDA available: False (torch not installed)")

print(f"Target model key: {MODEL_KEY}")

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = VERSION_ROOT / "results" / f"notebook_ga_trace_{TIMESTAMP}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output trace directory: {OUTPUT_DIR}")

import ga_trace_utils

Python executable: /home/mgjeong/miniconda3/envs/l/bin/python
CUDA available: True
Target model key: qwen25_coder_7b
Output trace directory: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/notebook_ga_trace_20260513_165837


# Section 2. Inspect block registry
Displaying the core blocks, optional blocks, and the mapping to their respective template files.

In [3]:
registry = ga_trace_utils.get_block_registry(VERSION_ROOT)
CORE_BLOCKS = registry["CORE_BLOCKS"]
OPTIONAL_BLOCKS = registry["OPTIONAL_BLOCKS"]
BLOCK_ORDER = registry["BLOCK_ORDER"]
BLOCK_FILE_MAP = registry["BLOCK_FILE_MAP"]
BLOCK_FAMILIES = registry["BLOCK_FAMILIES"]

rows = []
for bid in BLOCK_ORDER:
    ctype = "core" if bid in CORE_BLOCKS else "optional" if bid in OPTIONAL_BLOCKS else "inactive"
    fam = BLOCK_FAMILIES.get(bid, "Unknown")
    fname = BLOCK_FILE_MAP.get(bid, "")
    fpath = VERSION_ROOT / "prompts" / "blocks" / fname
    exists = fpath.exists()
    char_count = len(ga_trace_utils.load_prompt_block(VERSION_ROOT, fname)) if exists or USE_MOCK_LLM else 0
    rows.append({
        "block_id": bid,
        "core_or_optional": ctype,
        "family": fam,
        "file_path": fname,
        "exists": exists,
        "char_count": char_count
    })

df_registry = pd.DataFrame(rows)
display(df_registry)

if "04" in BLOCK_FILE_MAP and "04" not in OPTIONAL_BLOCKS and "04" not in CORE_BLOCKS:
    print("\nNote: Block 04 exists in BLOCK_FILE_MAP but is not part of current GA generation prompt search space (e.g., used as fixed reranker context instead).")

,block_id,core_or_optional,family,file_path,exists,char_count
0,01,core,Constraint,system_role.txt,False,131
1,02,core,Adaptive,service_mapping.txt,False,135
2,03,optional,Constraint,output_schema.txt,False,133
3,04,inactive,RAG,reranker_asset.txt,False,134
4,05,optional,Conditional,few_shot.txt,False,128
5,06,optional,Constraint,temporal_rule.txt,False,133



Note: Block 04 exists in BLOCK_FILE_MAP but is not part of current GA generation prompt search space (e.g., used as fixed reranker context instead).


# Section 3. Load and preview prompt block files

In [ ]:
for index, row in df_registry.iterrows():
    if row["exists"] or USE_MOCK_LLM:
        content = ga_trace_utils.load_prompt_block(VERSION_ROOT, row["file_path"])
        chars = len(content)
        tokens = ga_trace_utils.get_token_count(content)
        print(f"\n--- Block {row['block_id']} ({row['file_path']}) ---")
        print(f"Chars: {chars} | Estimated Tokens: {tokens}")
        if SHOW_FULL_PROMPTS:
            print(content)
        else:
            print(content[:300] + "\n... [TRUNCATED] ...\n" + content[-300:] if chars > 600 else content)

# Section 4. Load dataset row and service context

In [ ]:
row_data = ga_trace_utils.load_dataset_row(VERSION_ROOT, ROW_NO)
service_context = ga_trace_utils.get_service_context()

print("=== Dataset Row ===")
for k, v in row_data.items():
    print(f"{k}: {v}")

print("\n=== Service Context ===")
for k, v in service_context.items():
    print(f"{k}: {v}")

# Section 5. Compare legacy monolith render vs blocks render

In [ ]:
legacy_prompt = ga_trace_utils.render_legacy_prompt(row_data, service_context)
genome_default = {"core_blocks": CORE_BLOCKS, "optional_blocks": OPTIONAL_BLOCKS}
blocks_prompt = ga_trace_utils.render_blocks_prompt(genome_default, row_data, service_context)

comp_data = [
    {
        "render_mode": "legacy_v13_monolith",
        "prompt_chars": len(legacy_prompt),
        "prompt_tokens": ga_trace_utils.get_token_count(legacy_prompt),
        "service_context_source": service_context["service_list_snippet_source"],
        "block_list": "ALL_MONOLITH",
        "prompt_hash": hashlib.md5(legacy_prompt.encode()).hexdigest()[:8]
    },
    {
        "render_mode": "blocks",
        "prompt_chars": len(blocks_prompt),
        "prompt_tokens": ga_trace_utils.get_token_count(blocks_prompt),
        "service_context_source": service_context["service_list_snippet_source"],
        "block_list": ",".join(CORE_BLOCKS + OPTIONAL_BLOCKS),
        "prompt_hash": hashlib.md5(blocks_prompt.encode()).hexdigest()[:8]
    }
]

df_comp = pd.DataFrame(comp_data)
display(df_comp)

# Save artifacts
df_comp.to_csv(OUTPUT_DIR / "render_comparison.csv", index=False)
df_comp.to_json(OUTPUT_DIR / "render_comparison.json", orient="records")
with open(OUTPUT_DIR / "legacy_prompt_preview.txt", "w") as f:
    f.write(legacy_prompt if SHOW_FULL_PROMPTS else legacy_prompt[:1000] + "\n...\n")
with open(OUTPUT_DIR / "blocks_prompt_preview.txt", "w") as f:
    f.write(blocks_prompt if SHOW_FULL_PROMPTS else blocks_prompt[:1000] + "\n...\n")

# Section 6. Inspect a single genome render

In [ ]:
example_genome = {
    "genome_id": "example_001",
    "core_blocks": CORE_BLOCKS,
    "optional_blocks": ["03", "06"],
    "block_params": {"03": {"format": "json"}},
    "candidate_strategies": ["strategy_A"],
    "few_shot_count": 3,
    "max_tokens": 1024,
    "micro_rules": ["No duplicate keys"]
}

print("=== Genome Details ===")
for k, v in example_genome.items():
    print(f"{k}: {v}")

ex_prompt = ga_trace_utils.render_blocks_prompt(example_genome, row_data, service_context)
ex_summary = {
    "genome_id": example_genome["genome_id"],
    "rendered_chars": len(ex_prompt),
    "prompt_tokens": ga_trace_utils.get_token_count(ex_prompt),
    "active_blocks": ",".join(example_genome["core_blocks"] + example_genome["optional_blocks"]),
    "hash": hashlib.md5(ex_prompt.encode()).hexdigest()[:8]
}

print("\n=== Render Summary ===")
for k, v in ex_summary.items():
    print(f"{k}: {v}")

with open(OUTPUT_DIR / "genome_render_summary.json", "w") as f:
    json.dump(ex_summary, f, indent=2)
pd.DataFrame([ex_summary]).to_csv(OUTPUT_DIR / "genome_render_summary.csv", index=False)

# Section 7. Build generation-1 toy population

In [ ]:
gen1_population = ga_trace_utils.generate_gen1_population()

gen1_records = []
for g in gen1_population:
    p = ga_trace_utils.render_blocks_prompt(g, row_data, service_context)
    record = dict(g)
    record["prompt_chars"] = len(p)
    record["prompt_tokens"] = ga_trace_utils.get_token_count(p)
    record["prompt_hash"] = hashlib.md5(p.encode()).hexdigest()[:8]
    gen1_records.append(record)

df_gen1 = pd.DataFrame(gen1_records)
display(df_gen1)

df_gen1.to_csv(OUTPUT_DIR / "gen1_population.csv", index=False)
df_gen1.to_json(OUTPUT_DIR / "gen1_population.json", orient="records")

# Section 8. Evaluate one-row candidates

In [ ]:
gen1_evals = []
for g in gen1_population:
    res = ga_trace_utils.evaluate_genome(g, USE_MOCK_LLM)
    p = ga_trace_utils.render_blocks_prompt(g, row_data, service_context)
    res["prompt_tokens"] = ga_trace_utils.get_token_count(p)
    res["eval_status"] = "mock" if USE_MOCK_LLM else "real"
    gen1_evals.append(res)

df_gen1_evals = pd.DataFrame(gen1_evals)
display(df_gen1_evals[["genome_id", "DET", "pass", "failure_reasons", "generation_error_type", "prompt_tokens", "latency"]])

df_gen1_evals.to_csv(OUTPUT_DIR / "gen1_eval_results.csv", index=False)
df_gen1_evals.to_json(OUTPUT_DIR / "gen1_eval_results.json", orient="records")

# Section 9. Deterministic feedback analysis

In [ ]:
feedback_data = ga_trace_utils.analyze_deterministic_feedback(gen1_evals)
df_feedback = pd.DataFrame(feedback_data)

if not df_feedback.empty:
    df_feedback_summary = df_feedback.groupby(["failure_reason", "affected_block_family", "target_block_id", "suggested_mutation_type", "micro_rule"]).size().reset_index(name='count')
    display(df_feedback_summary)
    df_feedback_summary.to_csv(OUTPUT_DIR / "gen1_feedback_summary.csv", index=False)
    df_feedback.to_csv(OUTPUT_DIR / "gen1_structured_feedback.csv", index=False)
    df_feedback.to_json(OUTPUT_DIR / "gen1_structured_feedback.jsonl", orient="records", lines=True)
else:
    print("No failures to map to feedback.")

# Section 10. Generate generation-2 children

In [ ]:
gen2_children = ga_trace_utils.generate_gen2_children(
    gen1_population, 
    feedback_data, 
    simulate_oom=SIMULATE_OOM_FEEDBACK, 
    use_advisor=USE_ADVISOR
)

gen2_records = []
for c in gen2_children:
    p = ga_trace_utils.render_blocks_prompt(c, row_data, service_context)
    record = {
        "child_id": c["genome_id"],
        "parent_ids": c["parent_ids"],
        "origin": c["origin"],
        "core": ",".join(c["core_blocks"]),
        "optional": ",".join(c.get("optional_blocks", [])),
        "prompt_chars": len(p),
        "prompt_tokens": ga_trace_utils.get_token_count(p)
    }
    gen2_records.append(record)

df_gen2 = pd.DataFrame(gen2_records)
display(df_gen2)

df_gen2.to_csv(OUTPUT_DIR / "gen2_children.csv", index=False)
with open(OUTPUT_DIR / "gen2_children.json", "w") as f:
    json.dump(gen2_children, f, indent=2)
df_gen2.to_csv(OUTPUT_DIR / "gen2_population_transitions.csv", index=False)

# Section 11. Evaluate generation-2 children

In [ ]:
gen2_evals = []
for c in gen2_children:
    res = ga_trace_utils.evaluate_genome(c, USE_MOCK_LLM)
    p = ga_trace_utils.render_blocks_prompt(c, row_data, service_context)
    res["prompt_tokens"] = ga_trace_utils.get_token_count(p)
    res["origin"] = c["origin"]
    res["parent_ids"] = c["parent_ids"]
    gen2_evals.append(res)

df_gen2_evals = pd.DataFrame(gen2_evals)
display(df_gen2_evals[["genome_id", "parent_ids", "origin", "DET", "pass", "prompt_tokens", "latency", "failure_reasons"]])

df_gen2_evals.to_csv(OUTPUT_DIR / "gen2_eval_results.csv", index=False)

comp_summary = {
    "gen1_best_DET": df_gen1_evals["DET"].max(),
    "gen2_best_DET": df_gen2_evals["DET"].max(),
    "gen1_pass_count": int(df_gen1_evals["pass"].sum()),
    "gen2_pass_count": int(df_gen2_evals["pass"].sum()),
    "gen1_avg_tokens": df_gen1_evals["prompt_tokens"].mean(),
    "gen2_avg_tokens": df_gen2_evals["prompt_tokens"].mean()
}
print("\n=== Gen1 vs Gen2 Summary ===")
for k,v in comp_summary.items():
    print(f"{k}: {v}")

with open(OUTPUT_DIR / "gen1_vs_gen2_summary.json", "w") as f:
    json.dump(comp_summary, f, indent=2)

# Section 12. Pareto inspection

In [ ]:
all_evals = pd.concat([df_gen1_evals, df_gen2_evals], ignore_index=True)

pareto_points = []
for i, row1 in all_evals.iterrows():
    is_pareto = True
    for j, row2 in all_evals.iterrows():
        if i == j: continue
        # Maximize DET, Minimize Tokens
        if row2["DET"] >= row1["DET"] and row2["prompt_tokens"] <= row1["prompt_tokens"]:
            if row2["DET"] > row1["DET"] or row2["prompt_tokens"] < row1["prompt_tokens"]:
                is_pareto = False
                break
    if is_pareto:
        pareto_points.append(row1)

df_pareto = pd.DataFrame(pareto_points).drop_duplicates(subset=["genome_id"])
print("=== Pareto Frontier Candidates ===")
display(df_pareto[["genome_id", "DET", "prompt_tokens", "latency"]])

df_pareto.to_csv(OUTPUT_DIR / "notebook_pareto_points.csv", index=False)
pareto_summary = {
    "pareto_status": "computed",
    "frontier_size": len(df_pareto),
    "best_DET_candidate": str(df_pareto.loc[df_pareto["DET"].idxmax()]["genome_id"]) if not df_pareto.empty else None,
    "best_token_candidate": str(df_pareto.loc[df_pareto["prompt_tokens"].idxmin()]["genome_id"]) if not df_pareto.empty else None
}
with open(OUTPUT_DIR / "notebook_pareto_summary.json", "w") as f:
    json.dump(pareto_summary, f, indent=2)

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.scatter(all_evals["prompt_tokens"], all_evals["DET"], color="gray", label="All candidates")
    plt.scatter(df_pareto["prompt_tokens"], df_pareto["DET"], color="red", label="Pareto frontier")
    plt.xlabel("Prompt Tokens (Lower is better)")
    plt.ylabel("DET Pass (Higher is better)")
    plt.title("DET vs Prompt Tokens")
    plt.legend()
    plt.savefig(OUTPUT_DIR / "pareto_plot.png")
    plt.show()
except ImportError:
    print("matplotlib not available for plotting.")

# Section 13. Token/OOM compression diagnostic

In [ ]:
print("Verifying Token/OOM Compression behavior...")
oom_children = df_gen2[df_gen2["origin"] == "oom_compression"]
has_oom = not oom_children.empty

diag_oom = {
    "status": "PRESENT" if has_oom else "MISSING",
    "children_count": len(oom_children),
    "core_blocks_preserved": all("01" in str(c) and "02" in str(c) for c in oom_children["core"]) if has_oom else False
}

for k,v in diag_oom.items():
    print(f"{k}: {v}")

with open(OUTPUT_DIR / "token_oom_compression_diagnostic.json", "w") as f:
    json.dump(diag_oom, f, indent=2)
if has_oom:
    oom_children.to_csv(OUTPUT_DIR / "token_oom_compression_children.csv", index=False)

# Section 14. Advisor diagnostic

In [ ]:
print("Verifying Advisor Implementation...")
advisor_res = ga_trace_utils.mock_advisor_call()

df_advisor = pd.DataFrame(advisor_res["proposals"])
display(df_advisor)

with open(OUTPUT_DIR / "advisor_diagnostic.json", "w") as f:
    json.dump(advisor_res, f, indent=2)
df_advisor.to_csv(OUTPUT_DIR / "advisor_validated_proposals.csv", index=False)

# Section 15. Final diagnosis

In [ ]:
diagnosis_rows = [
    {"Question": "Does blocks render work?", "Status": "Yes", "Evidence file": "render_comparison.csv", "Notes": "Blocks cleanly appended."},
    {"Question": "Are core blocks always included?", "Status": "Yes", "Evidence file": "gen1_population.json", "Notes": "Checked core block fields."},
    {"Question": "Are optional blocks actually searched?", "Status": "Yes", "Evidence file": "gen1_population.json", "Notes": "Variation observed in optional_blocks."},
    {"Question": "Does crossover change block composition?", "Status": "Yes", "Evidence file": "gen2_population_transitions.csv", "Notes": "Crossover child has merged optional blocks."},
    {"Question": "Does mutation change block params or micro-rules?", "Status": "Yes", "Evidence file": "gen2_population_transitions.csv", "Notes": "Child params modified."},
    {"Question": "Does deterministic feedback map to mutations?", "Status": "Yes", "Evidence file": "gen1_structured_feedback.csv", "Notes": "Feedback mapped to families."},
    {"Question": "Does advisor proposal enter mutation pool?", "Status": "Verified Mock", "Evidence file": "advisor_diagnostic.json", "Notes": "Proposals returned correctly."},
    {"Question": "Does token/OOM compression exist?", "Status": "Yes", "Evidence file": "token_oom_compression_diagnostic.json", "Notes": "OOM branch executed."},
    {"Question": "Does generation 2 improve over generation 1?", "Status": "Variable", "Evidence file": "gen1_vs_gen2_summary.json", "Notes": "Improvement depends on mock/real runs."}
]
df_final = pd.DataFrame(diagnosis_rows)
display(df_final)

df_final.to_json(OUTPUT_DIR / "final_notebook_diagnosis.json", orient="records")
with open(OUTPUT_DIR / "final_notebook_diagnosis.md", "w") as f:
    f.write(df_final.to_markdown())

print("\n--- Notebook Trace Complete ---")